# Huấn luyện mô hình cho hệ thống Bãi đỗ xe thông minh

Notebook này huấn luyện **2 mô hình** mà `app.py` cần, rồi cho bạn tải về:

| File | Mô hình | Dữ liệu | Thời gian (GPU T4) |
|---|---|---|---|
| `yolov8s_plate.pt` | YOLOv8s phát hiện biển số | 3.132 ảnh biển số VN (Roboflow, CC BY 4.0) | ~40–60 phút |
| `crnn_bilstm_ctc.pth` | CNN + BiLSTM + CTC đọc ký tự | Ảnh biển số VN tổng hợp (+ ảnh thật nếu có) | ~30–40 phút |

**Trước khi chạy:** menu **Runtime → Change runtime type → T4 GPU**. Sau đó chạy lần lượt từng ô (Shift+Enter).


## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi -L

## 2. Tải project lên
Chọn file `parking_alpr.zip` trên máy bạn.

In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall('/content')
os.chdir('/content/parking_alpr')
print(os.listdir('.'))

## 3. Cài thư viện

In [ ]:
!pip -q install ultralytics roboflow pyyaml

## 4. Huấn luyện YOLOv8s — phát hiện biển số

Cần **Roboflow API key miễn phí**: đăng ký tại [roboflow.com](https://roboflow.com) → góc trái chọn workspace → **Settings → API Keys** → copy *Private API Key*.

Cấu hình giống báo cáo: ảnh 640×640, batch 16, AdamW, patience 20. Mặc định 60 epoch cho vừa thời lượng Colab miễn phí (báo cáo dùng 100 — tăng `--epochs` nếu bạn có thời gian).

In [ ]:
from getpass import getpass
RF_KEY = getpass('Dán Roboflow API key rồi Enter: ')

In [ ]:
!python training/train_yolo.py --roboflow-key "$RF_KEY" --epochs 60

Xem đường cong huấn luyện và kết quả trên tập validation:

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png', 'val_batch0_pred.jpg']:
    p = f'training/runs/plate_det/{f}'
    if os.path.exists(p):
        print(f); display(Image(p, width=900))

## 5. Huấn luyện OCR CNN + BiLSTM + CTC

Dùng đúng class `CRNN` trong `core/ocr_model.py` của app → nạp vào app chắc chắn khớp.
Mỗi epoch sinh mới 20.000 ảnh biển số tổng hợp (1 dòng / 2 dòng, ô tô / xe máy, nền trắng / vàng / xanh, nghiêng, mờ, nhiễu, thiếu sáng…).

Xem trước dữ liệu tổng hợp:

In [ ]:
!cd training && python plate_synth.py
display(Image('training/preview_synth.jpg'))

**(Tuỳ chọn, rất nên làm)** Nếu có ảnh biển số **thật** đã cắt sẵn: tạo thư mục `real_crops/` gồm ảnh + `labels.csv` (2 cột `filename,label`, ví dụ `img001.jpg,29AC88688`), nén zip, tải lên rồi thêm `--real-dir real_crops` vào lệnh huấn luyện. Chỉ vài trăm ảnh thật cũng giúp mô hình đọc ảnh camera tốt hơn đáng kể, và khi đó chỉ số đánh giá được tính trên ảnh thật.

In [ ]:
!python training/train_ocr.py --epochs 40 --workers 2

Vẽ quá trình huấn luyện OCR:

In [ ]:
import torch, matplotlib.pyplot as plt
ck = torch.load('weights/crnn_bilstm_ctc.pth', map_location='cpu', weights_only=False)
h = ck['history']
ep = [x['epoch'] for x in h]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [x['loss'] for x in h], color='#2a78d6', lw=2); ax[0].set_title('CTC loss (train)'); ax[0].set_xlabel('Epoch')
ax[1].plot(ep, [x['CharAcc'] for x in h], color='#2a78d6', lw=2, label='Character Accuracy')
ax[1].plot(ep, [x['PlateAcc'] for x in h], color='#eb6834', lw=2, label='Plate Accuracy')
ax[1].set_title('Độ chính xác (validation)'); ax[1].set_xlabel('Epoch'); ax[1].legend()
for a in ax: a.grid(alpha=.2); a.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()
print('Mô hình tốt nhất — epoch', ck['epoch'], ck['metrics'])

## 6. Chạy thử toàn bộ pipeline (YOLOv8 → cắt biển → OCR) trên ảnh test

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
from core.pipeline import ALPRPipeline, draw
from core import plate_utils
pipe = ALPRPipeline()
test_imgs = sorted(glob.glob('training/datasets/*/test/images/*'))[:12]
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, p in zip(axes.flat, test_imgs):
    img = cv2.imread(p)
    res = pipe.process(img)
    ax.imshow(cv2.cvtColor(draw(img, res), cv2.COLOR_BGR2RGB)); ax.axis('off')
    ax.set_title(', '.join(plate_utils.format_plate(r.plate) for r in res) or '(không thấy biển)', fontsize=11)
plt.tight_layout(); plt.show()

## 7. Tải 2 file trọng số về máy
Giải nén vào thư mục `parking_alpr/weights/` trên máy bạn rồi chạy lại `python app.py`.

In [ ]:
!cd weights && zip -q -j /content/weights.zip yolov8s_plate.pt crnn_bilstm_ctc.pth && ls -lh
files.download('/content/weights.zip')

**Mẹo:** Colab xoá dữ liệu khi hết phiên. Muốn giữ lại, lưu sang Google Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/parking_alpr_weights && cp weights/*.p* /content/drive/MyDrive/parking_alpr_weights/ && cp -r training/runs /content/drive/MyDrive/parking_alpr_weights/